In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('Breast Cancer METABRIC.csv')
target_col = 'Tumor Stage'
original_count = len(df)
df = df[df[target_col] != 4].copy()
removed_count = original_count - len(df)
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)
for i, feat in enumerate(removed_leakage, 1):
    print(f"  {i}. {feat}")

# Removal of data leakage features
missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")
    for i, feat in enumerate(high_missing, 1):
        print(f"  {i}. {feat}")

exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

print(f"{len(feature_cols)} candidate features have been identified")
X = df[feature_cols].copy()
y = df[target_col].copy()


# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        median_val = X[col].median()
        missing_count = X[col].isnull().sum()
        X[col].fillna(median_val, inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        missing_count = X[col].isnull().sum()
        X[col].fillna(mode, inplace=True)



# Label encoding
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    for col in categorical_cols:
        le = LabelEncoder()
        unique_vals = X[col].unique()
        X[col] = le.fit_transform(X[col].astype(str))
        print(f"  {col}: {len(unique_vals)} unique values, encoded as {sorted(X[col].unique())}")
else:
    print("No categorical features require encoding")

print(f"\nFeature matrix: {X.shape}")
print(f"Target variable: {y.shape}")


# Feature selection
K_FEATURES = 15
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
    k=min(K_FEATURES, len(feature_cols))
)
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)
print(feature_scores.head(min(20, len(feature_scores))).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

print(f"\nSelected {len(selected_features)} features:")
for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<50} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

# Data splitting

class_counts = y.value_counts()
for stage, count in class_counts.sort_index().items():
    print(f"  Stage {stage}: {count} 例 ({count/len(y)*100:.1f}%)")
min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape} ({len(X_train)/len(X_selected)*100:.1f}%)")
print(f"Test set: {X_test.shape} ({len(X_test)/len(X_selected)*100:.1f}%)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#  SMOTE

min_class_samples = y_train.value_counts().min()
k_neighbors = min(5, min_class_samples - 1)

for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  Stage {s}: {c} 例 ({c/len(y_train_bal)*100:.1f}%)")

# GBM

gbm_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    subsample=1.0,
    max_features='sqrt',
    random_state=42,
    verbose=0
)

gbm_model.fit(X_train_bal, y_train_bal)

# Cross-validation
X_selected_scaled = scaler.transform(X_selected)
cv_scores = cross_val_score(gbm_model, X_selected_scaled, y, cv=5, scoring='accuracy')
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\naverage: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")


# Importance of selected features

print("\n" + "=" * 80)
print("精选特征在模型中的重要性")
print("=" * 80)

final_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': gbm_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nModel importance of these {len(selected_features)} features:")
print(final_importance.to_string(index=False))


# Model evaluation


y_train_pred = gbm_model.predict(X_train_scaled)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall    = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1        = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

y_test_pred = gbm_model.predict(X_test_scaled)

test_accuracy  = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall    = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1        = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

